In [1]:
from langchain_classic.retrievers import WikipediaRetriever

d:\Aakarsh\Coding\langchain\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
wikipedia_ret=WikipediaRetriever(top_k_results=3,lang='en')

In [3]:
query="What is age of Kanye West"

In [4]:
docs=wikipedia_ret.invoke(query)

In [6]:
for doc in docs:
    print(doc)
    print("*"*50)

page_content='American rapper Kanye West has received mainstream media attention for his outspoken views on numerous political and social issues. He donated to the political campaigns of Barack Obama in 2008 and 2012, and Hillary Clinton in 2016. From 2018 to 2020, West publicly endorsed Donald Trump on several occasions until launching his own campaign, and supported Trump once more in 2024. West has met with Trump on several occasions, most recently dining with Trump and live streamer Nick Fuentes in 2022. In his unsuccessful run for President of the United States in 2020, he espoused consistent life ethic and other Christian positions. His subsequent campaign in the 2024 election was active for several months in late 2022, although no formal paperwork was filed.
Beginning in 2018, West expressed opposition to abortion, capital punishment, and welfare, and supported gun rights and gay marriage. His views grew more radical following his divorce from Kim Kardashian. In December 2022, W

In [65]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI
embedding=GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [66]:
model=ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [8]:
from dotenv import load_dotenv
load_dotenv()

True

In [41]:
from langchain_classic.schema import Document

docs = [
    Document(page_content="Climate change is causing global temperatures to rise due to increasing greenhouse gas emissions from human activities."),
    
    Document(page_content="Global warming is mainly driven by carbon dioxide emissions from burning fossil fuels like coal, oil, and natural gas."),
    
    Document(page_content="Melting glaciers and rising sea levels are major effects of climate change, threatening coastal cities and ecosystems."),
    
    Document(page_content="Deforestation contributes to climate change by reducing the Earth's ability to absorb carbon dioxide from the atmosphere."),
    
    # slightly different but still somewhat related
    Document(page_content="Plastic pollution is a major environmental problem, harming marine life and increasing waste in oceans and landfills."),
]


In [13]:
from langchain_classic.vectorstores import FAISS
import faiss
from langchain_classic.docstore import InMemoryDocstore

In [28]:
dim=len(embedding.embed_query("Hello"))

In [29]:
faiss_store=FAISS(embedding_function=embedding,
                  index=faiss.IndexFlatL2(dim),
                  docstore=InMemoryDocstore(),
                  index_to_docstore_id={})

In [42]:
faiss_store.add_documents(docs)

['6c2ca81b-32f9-4651-8583-bc2ae45db9e1',
 '61a83495-64ef-4b59-b7f2-6fbde2c2db8a',
 '9ae06fa8-5595-4b2a-a6fd-12a82b259e35',
 'e56057b2-7d74-414a-b45f-d9863e417875',
 'd1b6ed49-603e-4fbc-a056-d32fcb98ac90']

In [57]:
vector_ret=faiss_store.as_retriever(search_type="mmr",search_kwargs={"k":3,'lambda_mult':0.8})

In [58]:
query="Global Warming"

In [61]:
result=vector_ret.invoke(query)

In [62]:
for i in result:
    print(i)

page_content='Climate change is causing global temperatures to rise due to increasing greenhouse gas emissions from human activities.'
page_content='Global warming is mainly driven by carbon dioxide emissions from burning fossil fuels like coal, oil, and natural gas.'
page_content='Melting glaciers and rising sea levels are major effects of climate change, threatening coastal cities and ecosystems.'


In [68]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [76]:
from langchain_classic.schema import Document

new_docs = [
    # ----- First 5: Energy systems + Fitness/Health -----
    Document(page_content="A balanced diet supports the body's energy system by providing carbohydrates, fats, and proteins needed for daily functioning and workouts."),
    
    Document(page_content="Carbohydrates are the primary energy source for the body, especially during high-intensity exercise, as they are quickly converted into glucose."),
    
    Document(page_content="Healthy fats play an important role in long-term energy production, hormone regulation, and absorption of vitamins like A, D, E, and K."),
    
    Document(page_content="Protein is essential for muscle repair and recovery, helping rebuild tissues after exercise and supporting overall metabolic health."),
    
    Document(page_content="Micronutrients like iron, magnesium, and vitamin B12 are crucial for energy metabolism, oxygen transport, and reducing fatigue in the body."),

    # ----- Next 5: Random but still related to energy in some sense -----
    Document(page_content="Solar power systems generate renewable energy by converting sunlight into electricity using photovoltaic panels."),
    
    Document(page_content="Electric vehicles rely on battery energy storage systems to power motors and improve transportation efficiency."),
    
    Document(page_content="A power grid is a large-scale energy distribution system that delivers electricity from generators to consumers through transmission networks."),
    
    Document(page_content="Wind turbines produce energy by converting the kinetic energy of wind into electrical energy using rotating blades."),
    
    Document(page_content="In physics, energy can exist in many forms such as kinetic, thermal, chemical, and electrical energy, and it is always conserved."),
]


In [80]:
new_vs=FAISS.from_documents(documents=new_docs,embedding=embedding)

In [77]:
mq_ret=MultiQueryRetriever.from_llm(
    retriever=new_vs.as_retriever(search_kwargs={"k":2}),
    llm=model
)

In [73]:
query="How to improve energy levels and maintain balance"

In [81]:
results=mq_ret.invoke(query)

In [82]:
for i in results:
    print(i)

page_content='Sleep is essential for restoring energy systems in the body, improving hormonal balance, and enhancing muscle recovery after training.'
page_content='Proper nutrition supports the body's energy systems by supplying glucose, electrolytes, and proteins needed for recovery and muscle repair.'
page_content='Aerobic energy systems provide long-lasting energy for endurance workouts by using oxygen to break down glucose and fat for sustained performance.'
